# GCN Example (PyTorch) — Node Classification on the Karate Club Graph

The same minimal **Graph Convolutional Network (GCN)** as `gcn_karate_club.ipynb`, but
implemented in **PyTorch** — from scratch, no graph libraries required.

We use **Zachary's Karate Club** (34 nodes). The task is *semi-supervised node
classification*: we only know the label of **2 nodes** and let the GCN propagate that
across the edges to classify everyone else.

A GCN layer is: $H' = \sigma(\hat{A}\,H\,W)$, where $\hat{A}$ is the normalized
adjacency matrix.

> Needs `torch`. Install with `pip install torch` if missing. Runs instantly on CPU.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## 1. Build the Karate Club graph

78 hard-coded edges (no `networkx` needed). Node 0 = instructor, node 33 = president.

In [ ]:
edges = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),(0,12),
    (0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),(1,17),(1,19),
    (1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),(3,7),
    (3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),(8,33),
    (9,33),(13,33),(14,32),(14,33),(15,32),(15,33),(18,32),(18,33),(19,33),
    (20,32),(20,33),(22,32),(22,33),(23,25),(23,27),(23,29),(23,32),(23,33),
    (24,25),(24,27),(24,31),(25,31),(26,29),(26,33),(27,33),(28,31),(28,33),
    (29,32),(29,33),(30,32),(30,33),(31,32),(31,33),(32,33),
]

N = 34
A = np.zeros((N, N), dtype='float32')
for i, j in edges:
    A[i, j] = 1.0
    A[j, i] = 1.0

print('Nodes:', N, '| Edges:', len(edges))

## 2. Normalize the adjacency matrix

$\hat{A} = \tilde{D}^{-1/2}\,\tilde{A}\,\tilde{D}^{-1/2}$ with $\tilde{A} = A + I$.

In [ ]:
A_tilde = A + np.eye(N, dtype='float32')
D = A_tilde.sum(axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
A_hat = (D_inv_sqrt @ A_tilde @ D_inv_sqrt).astype('float32')
A_hat = torch.tensor(A_hat, device=device)

print('Normalized adjacency shape:', tuple(A_hat.shape))

## 3. Features and labels

Identity (one-hot) features; only nodes 0 and 33 are labelled.

In [ ]:
X = torch.eye(N, device=device)

y_true = np.array([
    0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,1,0,1,0,1,1,
    1,1,1,1,1,1,1,1,1,1
])
y_true_t = torch.tensor(y_true, dtype=torch.long, device=device)

labeled_nodes = torch.tensor([0, 33], device=device)
y_labeled = y_true_t[labeled_nodes]
print('Labeled nodes:', labeled_nodes.tolist(), '-> labels', y_labeled.tolist())

## 4. Define a GCN layer and model

Each layer computes $\hat{A}\,(X W)$. The model stacks two of them (16 hidden units).

In [ ]:
class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)

    def forward(self, x, A_hat):
        return A_hat @ self.linear(x)   # A_hat (X W)

class GCN(nn.Module):
    def __init__(self, in_features, hidden=16, classes=2):
        super().__init__()
        self.gc1 = GCNLayer(in_features, hidden)
        self.gc2 = GCNLayer(hidden, classes)

    def forward(self, x, A_hat):
        x = F.relu(self.gc1(x, A_hat))
        self.embeddings = x             # keep hidden embeddings for plotting
        return self.gc2(x, A_hat)

model = GCN(in_features=N).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
print(model)

## 5. Train

Loss is computed only on the 2 labelled nodes; gradients flow to the whole graph
through the adjacency mixing.

In [ ]:
EPOCHS = 200

for epoch in range(1, EPOCHS + 1):
    model.train()
    logits = model(X, A_hat)
    loss = F.cross_entropy(logits[labeled_nodes], y_labeled)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            preds = model(X, A_hat).argmax(dim=1)
            acc = (preds == y_true_t).float().mean().item()
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={loss.item():.4f}  full-graph acc={acc:.3f}')

## 6. Results

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X, A_hat).argmax(dim=1).cpu().numpy()
acc = (preds == y_true).mean()
print(f'Final accuracy on all {N} nodes: {acc:.1%}')
print('Predicted factions:', preds)
print('True factions:     ', y_true)

## 7. Visualize the learned embeddings

16-dim hidden embeddings projected to 2D with PCA. Labelled nodes have a black border.

In [ ]:
with torch.no_grad():
    _ = model(X, A_hat)
    emb = model.embeddings.cpu().numpy()

emb_centered = emb - emb.mean(axis=0)
u, s, vt = np.linalg.svd(emb_centered, full_matrices=False)
coords = emb_centered @ vt[:2].T

plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=preds, cmap='coolwarm', s=200, edgecolors='gray')
for n in [0, 33]:
    plt.scatter(coords[n, 0], coords[n, 1], s=320, facecolors='none', edgecolors='black', linewidths=2.5)
for i, (x_, y_) in enumerate(coords):
    plt.text(x_, y_, str(i), ha='center', va='center', fontsize=8)
plt.title('GCN node embeddings (PCA) — colored by predicted faction')
plt.xlabel('PC 1'); plt.ylabel('PC 2')
plt.show()

### Things to try

- Change `hidden` units or `EPOCHS`.
- Label different/more nodes and observe the effect on accuracy.
- Add dropout between layers, or a third GCN layer.